In [ ]:
import os
import glob
from collections import defaultdict

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import spearmanr, wilcoxon

In [ ]:
titles = {
    'L2': r'$L_2$',
    'SSIM': 'SSIM',
    'MONAI_alex': 'AlexNet',
    'MONAI_vgg': 'VGG-16',
    'UltraPIPS_vit_imagenet': 'ViT',
    'UltraPIPS_swin_imagenet': 'SwinViT',
    'UltraPIPS_clip': 'CLIP',
    'MONAI_radimagenet_resnet50': 'RadImageNet',
    'UltraPIPS_medsam': 'MedSAM',
    'UltraPIPS_biomedclip': 'BiomedCLIP',
    'UltraPIPS_usfm': 'USFM',
    'UltraPIPS_tusa_vit': 'TUSA',
    'UltraPIPS_ultrasound_clip': 'Ultrasound-CLIP',
}

order = [
    r'$L_2$', 'SSIM',
    'AlexNet', 'VGG-16', 'ViT', 'SwinViT', 'CLIP',
    'RadImageNet', 'MedSAM', 'BiomedCLIP',
    'USFM', 'TUSA', 'Ultrasound-CLIP'
]

groups = {
    'Classical': [r'$L_2$', 'SSIM'],
    'ImageNet': ['AlexNet', 'VGG-16', 'ViT', 'SwinViT', 'CLIP'],
    'Radiology': ['RadImageNet', 'MedSAM', 'BiomedCLIP'],
    'Ultrasound': ['USFM', 'TUSA', 'Ultrasound-CLIP']
}

In [ ]:
def iterdir(src_dir, augs):
    conf_combined = defaultdict(dict)
    std_combined = defaultdict(dict)
    result_bank = defaultdict(dict)
    per_metric = defaultdict(list)

    for test in augs:
        raw_conf   = defaultdict(list)
        raw_metric = defaultdict(list)
        
        for result_file in glob.glob(os.path.join(src_dir, test, '**/*.npz'), recursive=True):
            data = np.load(result_file)
            metrics = [k for k in data.keys() if k not in ['y', 'yhat', 'confidence', 'variants']]

            variants = data['variants'][1:]
            indices = np.array(
                sorted(range(len(variants)), key=lambda x: float(variants[x].split('-')[-1]))
            )

            confidence = np.log(data['confidence'][:, 1:])[:, indices]

            for metric in metrics:
                raw_conf[metric].append(confidence)  # (N, V)
                raw_metric[metric].append(data[metric][:, indices])              # (N, V)

        for metric in raw_conf:
            conf_all = np.vstack(raw_conf[metric])
            met_all  = np.vstack(raw_metric[metric])

            r_values = np.array([
                spearmanr(conf_all[i], met_all[i]).statistic
                for i in range(conf_all.shape[0])
            ]).clip(-1 + 1e-7, 1 - 1e-7)

            conf_combined[test][titles[metric]] = abs(r_values.mean())
            std_combined[test][titles[metric]] = abs(r_values).std()
            result_bank[titles[metric]][test] = r_values
            per_metric[titles[metric]].append(r_values)

    # Aggregate
    df_mean = pd.DataFrame(conf_combined)
    df_mean = df_mean.loc[order, :]
    df_mean.columns = [c.title().replace('_', ' ') for c in conf_combined.keys()]

    df_std = pd.DataFrame(std_combined)
    df_std = df_std.loc[order, :]
    df_std.columns = [c.title().replace('_', ' ') for c in std_combined.keys()]

    # Convert to Latex
    latex_df = df_mean.copy()
    best_mask = df_mean.eq(df_mean.max(axis=0))

    for col in df_mean.columns:
        latex_df[col] = [f"$\\mathbf{{{m:.2f} \\pm {s:.2f}}}$" if best_mask.loc[row, col] else f"{m:.2f} $\\pm$ {s:.2f}" for row, m, s in zip(df_mean.index, df_mean[col], df_std[col])]

    return latex_df, per_metric, result_bank

In [ ]:
camus_df, camus_r, camus_bank = iterdir('../assets/results/01', ['depth', 'rotation', 'sector_width', 'translation'])
echonet_df, echonet_r, echonet_bank = iterdir('../assets/results/02', ['depth', 'rotation', 'sector_width', 'translation'])

In [ ]:
overall_mean, overall_std = {}, {}

best = 0
for k in camus_r:
    r = np.concat((np.concat(camus_r[k]), np.concat(echonet_r[k])))

    overall_mean[k] = abs(r.mean())
    overall_std[k] = abs(r).std()

    if overall_mean[k] > best:
        best = overall_mean[k]

overall = {}
for k in overall_mean:
    m, s = overall_mean[k], overall_std[k]

    if np.around(m, 2) == np.around(best, 2):
        overall[k] = f"$\\mathbf{{{m:.2f} \\pm {s:.2f}}}$"
    else:
        overall[k] = f"{m:.2f} $\\pm$ {s:.2f}"

In [ ]:
df2 = pd.concat({'CAMUS': camus_df, 'EchoNet': echonet_df}, axis=1)
df2.loc[list(overall_mean.keys()), 'Overall'] = overall


df2.insert(0, ('', 'Model'), camus_df.index)

n = len(camus_df.columns)
latex = df2.to_latex(
    label="tab:classification",
    float_format="%.2f",
    index=False,
    column_format=f"c|{'c'*n}|{'c'*n}|c",
    multicolumn_format="c|",
    caption="Correlation of EchoPrime view classifier confidence with distance metrics at each deformations"
)

lines = latex.replace(r'\begin{tabular}', r'\centering\resizebox{\textwidth}{!}{\begin{tabular}').replace(r'\end{tabular}', r'\end{tabular}}').split('\n')
data_start = [i for i, l in enumerate(lines) if '&' in l][-len(camus_df)]

for i in [10, 7, 2]:
    lines.insert(data_start + i, r'\midrule')

print('\n'.join(lines))

# Stats

In [ ]:
groups_results = {}

for group in groups:
    groups_results[group] = {}
    for aug in camus_bank['SSIM']:
        groups_results[group][aug] = []

def group_results_old(group_results, result_bank):

    for metric, augs in result_bank.items():
        group = [g for g, l in groups.items() if metric in l][0]

        for aug, r_values in augs.items():
            group_results[group][aug].append(r_values)

group_results_old(groups_results, camus_bank)

final_agg = defaultdict(list)
for group, augs in groups_results.items():
    for r_values in augs.values():
        final_agg[group].append(abs(np.mean(r_values)))

groups_results = {}

for group in groups:
    groups_results[group] = {}
    for aug in camus_bank['SSIM']:
        groups_results[group][aug] = []

group_results_old(groups_results, echonet_bank)
for group, augs in groups_results.items():
    for r_values in augs.values():
        final_agg[group].append(abs(np.mean(r_values)))

In [ ]:
for group in ['Classical', 'ImageNet', 'Radiology']:
    p = wilcoxon(final_agg[group], final_agg['Ultrasound']).pvalue
    print(f'{group}: {p}')

# Plots

In [ ]:
colors = {
    'Classical': '#34495e',
    'ImageNet': '#3498db',
    'Radiology': '#e67e22',
    'Ultrasound':  '#2ecc71'
}

patterns = ['', '//', '\\', '..', 'xx', '\\\\', ]

plt.style.use('seaborn-v0_8-whitegrid')
plot_titles = ['CAMUS', 'EchoNet']

for result_bank, plot_title in zip([camus_bank, echonet_bank], plot_titles):
    fig, ax = plt.subplots(figsize=(10, 6))

    df = {}
    for model, augs in result_bank.items():
        group = [g for g, l in groups.items() if model in l][0]
        color = colors[group] 

        r_values = np.concat(list(augs.values()))
        
        r = abs(r_values.mean())
        lo = hi = abs(r_values).std()

        df[model] = {
            'r': r,
            'lo': abs(lo-r),
            'hi': abs(hi-r),
            'color': color,
            'group_idx': list(groups.keys()).index(group)
        }


    df = pd.DataFrame.from_dict(df)
    df = df[order].T

    bars = ax.barh(df.index, df.r, color=df.color, edgecolor='0.2', alpha=0.8)

    last_group = -1
    for bar, group_idx in zip(bars, df.group_idx):
        if group_idx == last_group:
            pattern_id += 1
        else:
            pattern_id = 0
            last_group = group_idx
        bar.set_hatch(patterns[pattern_id])


    for i, (m, lo, hi) in enumerate(zip(df.r, df.lo, df.hi)):
        ax.errorbar(m, i, xerr=[[lo], [hi]], fmt='none', ecolor=df.color.iloc[i], capsize=4, elinewidth=2)

    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.tick_params(axis='both', labelsize=16) # Adjust labelsize as needed
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()

    i = '01' if 'CAMUS' in plot_title else '02'
    plt.savefig(f'../assets/results/{i}/echoprime.png')
plt.show()
